# 🌍 Periliminal.Space × LingBot-World v2
## AI Reference World Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joeholloway445-maker/CATSINO.CASINO/blob/main/pipeline/LingBot_World_Colab_Pipeline.ipynb)

**What this does:** Generates 10 AI-powered reference videos showing what each Periliminal.Space reality layer and DFW city should look like. Runs entirely in Google's cloud — never touches your local files.

**⏱ Total time:** ~30-90 minutes (all 10 environments)

**How it works:**
1. Installs LingBot-World v2 (Robbyant's open-source world model)
2. Downloads the 14B causal-fast model
3. Generates reference videos for 10 environments
4. Packages everything into a zip for download

---

### ✅ One-click instructions

| Step | Action |
|------|--------|
| 1 | **Runtime → Change runtime type → T4 GPU** ☝️ Do this first! |
| 2 | Click **Runtime → Run all** (or Ctrl+F9) |
| 3 | Wait ~30-90 min. Each env shows ✅ when done. |
| 4 | The zip downloads automatically. Extract into `assets/references/lingbot/` |
| 5 | Press **F5** in Godot — reference videos appear as skyboxes |

---

In [ ]:
# Cell 1: Verify GPU
import torch, os, sys, math, time, subprocess, json, zipfile, numpy as np
from PIL import Image, ImageDraw
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError('\n\n❌ NO GPU DETECTED!\nGo to Runtime \u2192 Change runtime type \u2192 T4 GPU, then restart.')

gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f'\u2705 GPU: {gpu_name} ({vram:.1f} GB VRAM)')
os.chdir('/content')

---
## Step 1: Install LingBot-World v2
Clones the repo, installs dependencies. Takes ~3-5 min.

In [ ]:
print('Cloning LingBot-World v2...')
if not os.path.exists('/content/lingbot-world-v2/generate.py'):
    !git clone --quiet https://github.com/Robbyant/lingbot-world-v2.git 2>&1 | tail -1
    %cd /content/lingbot-world-v2
    !pip install -q -r requirements.txt 2>&1 | tail -1
    !pip install -q flash-attn --no-build-isolation 2>&1 | tail -1
    !pip install -q huggingface_hub[cli] 2>&1 | tail -1
else:
    %cd /content/lingbot-world-v2
    print('Already installed')
print('\u2705 LingBot-World v2 ready')

---
## Step 2: Download Model (14B parameters)
Downloads the causal-fast model (~8GB). Takes ~5-10 min.

In [ ]:
MODEL_DIR = '/content/models/lingbot-world-v2-14b-causal-fast'
os.makedirs(MODEL_DIR, exist_ok=True)

model_size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(MODEL_DIR) for f in fn) if os.path.exists(MODEL_DIR) else 0
if model_size < 7_000_000_000:
    print('Downloading model (this is the big one - ~8GB)...')
    !huggingface-cli download robbyant/lingbot-world-v2-14b-causal-fast --local-dir {MODEL_DIR} --local-dir-use-symlinks False --quiet 2>&1
    model_size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, fn in os.walk(MODEL_DIR) for f in fn)
else:
    print('Model already downloaded')

print(f'\u2705 Model: {model_size / 1e9:.1f} GB')

---
## Step 3: Define All 10 Environments
Each environment has a visual prompt, base image color, frame count, and camera motion.

In [ ]:
ENVIRONMENTS = [
    {'id': 'subliminal', 'name': 'Subliminal \u2014 The Apartment',
     'prompt': 'A warm cat-themed apartment with purple neon ambient lighting, cozy bed, holographic cat posters on walls, window showing a starry void beyond, clean zen-like interior with soft glowing edges, first-person view standing at the center of the room, photorealistic UE5 quality',
     'color': (40, 25, 55), 'frames': 81, 'motion': 'pan right slowly'},
    {'id': 'liminal', 'name': 'Liminal \u2014 The Between',
     'prompt': 'An endless void between spaces, Victorian-style hallway floating in darkness with impossible geometry, brass fixtures floating unsupported, doors appearing and dissolving in the mist, warm gas lamps casting pools of light, deep indigo fog below, surreal and beautiful liminal space, first-person wandering, photorealistic',
     'color': (20, 15, 30), 'frames': 81, 'motion': 'walk forward slowly'},
    {'id': 'hyperliminal', 'name': 'Hyperliminal \u2014 The Catsino',
     'prompt': 'A massive cat-themed casino interior, golden chandeliers shaped like paw prints, neon signs in cat paw patterns, roulette tables with cat dealers, slot machines with animated cat reels, red velvet carpets, art deco architecture with cat-eared arches, bustling with anthropomorphic cat characters, grand and luxurious, cinematic wide shot, photorealistic',
     'color': (80, 20, 30), 'frames': 81, 'motion': 'pan left slowly'},
    {'id': 'periliminal', 'name': 'Periliminal \u2014 The Gauntlet',
     'prompt': 'A surreal psychological landscape made of floating platforms and broken mirrors, swirling vortex of memories in the sky, shadowy figures at the edge of vision, reality bending and distorting, emotional weight visible in the air like shimmering heat waves, ground translucent showing stars beneath, disorienting and profound, first-person exploration, photorealistic',
     'color': (10, 5, 20), 'frames': 81, 'motion': 'crawl forward slowly'},
    {'id': 'extraliminal', 'name': 'Extraliminal \u2014 The Overlay',
     'prompt': 'A holographic augmented reality overlay layer over a real city, giant cat-shaped neon data streams flowing between buildings, augmented reality markers floating in the air like digital fireflies, transparent cyan blue holographic screens showing real-time battle data, guild banners made of pure light, pokemon-go style gym markers glowing in the distance, cyberpunk AR aesthetic, photorealistic',
     'color': (25, 40, 60), 'frames': 81, 'motion': 'orbit around central point'},
    {'id': 'dallas', 'name': 'New Dallas',
     'prompt': 'Futuristic Dallas skyline at golden hour, Reunion Tower redesigned as a neon cat spire, Bank of America Plaza with holographic cat face, wide tree-lined boulevards with flying cars, glass skyscrapers with integrated digital billboards showing cat animations, warm amber lighting, people in cyberpunk fashion walking below, photorealistic cityscape',
     'color': (60, 50, 40), 'frames': 121, 'motion': 'drone flyover from south to north'},
    {'id': 'fort_worth', 'name': "Hell's Half Acre",
     'prompt': "Hell's Half Acre - a cyberpunk Western hybrid city, historic stockyards transformed with neon cattle skull signs, wooden boardwalks with holographic advertisements, mechanical longhorns with glowing eyes, dust storms lit by purple neon, saloon bars with robot bartenders, Victorian-meets-cyberpunk architecture, gritty and atmospheric, photorealistic",
     'color': (55, 35, 25), 'frames': 121, 'motion': 'walk down main street'},
    {'id': 'denton', 'name': 'Sky Fjord',
     'prompt': 'Sky Fjord - a floating college town built into a cliff face, waterfalls cascading between hovering platforms, university buildings with glass domes, students on flying skateboards, the courthouse dome glowing with data streams, forest integration with the architecture, misty atmosphere with rays of light breaking through, serene and academic, photorealistic',
     'color': (45, 60, 70), 'frames': 121, 'motion': 'ascending gondola view'},
    {'id': 'arlington', 'name': 'Soulless Sanctuary',
     'prompt': "Soulless Sanctuary - a massive arena city with a giant spherical AT&T Stadium at center, a space elevator in the distance, college campus with holographic cat mascots, the Globe Life Field transformed into a gladiator pit, space station docked overhead, purple and teal neon everywhere, industrial-chic aesthetic, photorealistic wide shot",
     'color': (50, 40, 55), 'frames': 121, 'motion': 'orbit around stadium'},
    {'id': 'supraliminal_overview', 'name': 'DFW Metroplex Overview',
     'prompt': 'A sprawling futuristic DFW metroplex seen from above, four distinct city cores connected by maglev trains, neon rivers flowing between them, cat-shaped cloud formations in the sky, holographic billboards visible from miles away, mega-city with distinct districts each with their own color scheme, cinematic drone establishing shot, photorealistic',
     'color': (35, 45, 55), 'frames': 81, 'motion': 'helicopter flyover across skyline'},
]
print(f'\u2705 {len(ENVIRONMENTS)} environments defined')

---
## Step 4: Create Base Images + Camera Paths

In [ ]:
OUT_DIR = '/content/generated'
os.makedirs(f'{OUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUT_DIR}/videos', exist_ok=True)

def create_camera_path(num_frames, motion_type):
    fx, fy, cx, cy = 500, 500, 416, 240
    intrinsics = np.tile([fx, fy, cx, cy], (num_frames, 1)).astype(np.float32)
    poses = []
    for i in range(num_frames):
        t = i / max(num_frames - 1, 1)
        pose = np.eye(4, dtype=np.float32)
        if 'pan right' in motion_type: pose[0, 3] = t * 2.0
        elif 'pan left' in motion_type: pose[0, 3] = -t * 2.0
        elif 'walk forward' in motion_type: pose[2, 3] = -t * 3.0
        elif 'crawl forward' in motion_type: pose[1, 3] = -0.3; pose[2, 3] = -t * 1.5
        elif 'orbit' in motion_type:
            angle = t * 2 * math.pi
            pose[0, 3] = 5.0 * math.cos(angle)
            pose[2, 3] = 5.0 * math.sin(angle); pose[1, 3] = 1.0
        elif 'drone flyover' in motion_type or 'helicopter flyover' in motion_type:
            pose[0, 3] = t * 5.0; pose[1, 3] = 2.0 + t * 1.0; pose[2, 3] = -3.0
        elif 'walk down main street' in motion_type:
            pose[0, 3] = -t * 1.0; pose[2, 3] = -t * 4.0
        elif 'ascending gondola' in motion_type:
            pose[1, 3] = t * 4.0; pose[2, 3] = -2.0
        poses.append(pose)
    return intrinsics, np.stack(poses)

for env in ENVIRONMENTS:
    img = Image.new('RGB', (832, 480), env['color'])
    draw = ImageDraw.Draw(img)
    for y in range(480):
        f = y / 480
        draw.line([(0, y), (832, y)], fill=tuple(int(c * (0.5 + f * 0.5)) for c in env['color']))
    img.save(f'{OUT_DIR}/images/{env["id"]}.jpg', quality=85)
    intrinsics, poses = create_camera_path(env['frames'], env['motion'])
    action_dir = f'{OUT_DIR}/videos/{env["id"]}'
    os.makedirs(action_dir, exist_ok=True)
    np.save(f'{action_dir}/intrinsics.npy', intrinsics)
    np.save(f'{action_dir}/poses.npy', poses)

print(f'\u2705 Created {len(ENVIRONMENTS)} base images + camera paths')

---
## Step 5: Generate Videos with LingBot-World

This is the main generation step. Each environment takes ~3-8 min.

\ud83d\udccd **If Colab times out** (free tier disconnects after ~90 min idle):
Just re-run this cell — it skips already-generated videos.

\ud83d\udccd **To resume from a specific env:** Change `START_FROM` below.

In [ ]:
%cd /content/lingbot-world-v2
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

START_FROM = ENVIRONMENTS[0]['id']
started = False
total = len(ENVIRONMENTS)

for idx, env in enumerate(ENVIRONMENTS):
    env_id = env['id']
    if env_id == START_FROM: started = True
    if not started: continue
    
    video_path = f'{OUT_DIR}/videos/{env_id}.mp4'
    if os.path.exists(video_path) and os.path.getsize(video_path) > 100000:
        print(f'[{idx+1}/{total}] \u2705 {env["name"]} - already exists')
        continue
    
    print(f'[{idx+1}/{total}] \ud83c\udfac Generating {env["name"]} ({env["frames"]} frames)...')
    t0 = time.time()
    
    cmd = [
        'torchrun', '--nproc_per_node=1', 'generate.py',
        '--task', 'i2v-A14B',
        '--size', '480*832',
        '--ckpt_dir', MODEL_DIR,
        '--image', f'{OUT_DIR}/images/{env_id}.jpg',
        '--action_path', f'{OUT_DIR}/videos/{env_id}',
        '--frame_num', str(env['frames']),
        '--local_attn_size', '18',
        '--sink_size', '6',
        '--offload_model', 'True',
        '--base_seed', str(42 + idx),
        '--prompt', env['prompt'],
        '--save_file', video_path
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    elapsed = time.time() - t0
    
    if result.returncode == 0 and os.path.exists(video_path):
        size_mb = os.path.getsize(video_path) / 1e6
        print(f'  \u2705 Done in {elapsed:.0f}s ({size_mb:.1f} MB)')
    else:
        err = result.stderr[-300:] if result.stderr else 'unknown error'
        print(f'  \u274c Failed after {elapsed:.0f}s - {err}')
        print(f'  Continuing to next environment...')

print(f'\n{"="*50}')
print(f'\ud83c\udfc1 Generation complete!')
print(f'Videos saved to: {OUT_DIR}/videos/')

---
## Step 6: Package into Zip

In [ ]:
ZIP_NAME = '/content/periliminal_worlds.zip'
print(f'Packaging into {ZIP_NAME}...')

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(f'{OUT_DIR}/videos'):
        for file in files:
            if file.endswith('.mp4'):
                fp = os.path.join(root, file)
                zf.write(fp, f'videos/{file}')
                print(f'  \ud83d\udcc1 {file} ({os.path.getsize(fp)/1e6:.1f} MB)')

size_mb = os.path.getsize(ZIP_NAME) / 1e6
print(f'\n\u2705 Package created: {ZIP_NAME} ({size_mb:.1f} MB)')

---
## Step 7: Download

Click the button below OR find `periliminal_worlds.zip` in Colab's file browser (\ud83d\udcc1 icon in the left sidebar) → Right-click → Download.

In [ ]:
from google.colab import files

try:
    files.download(ZIP_NAME)
    print('\u2705 Download started!')
except Exception as e:
    print(f'Auto-download failed: {e}')
    print(f'\nManually download from Colab left sidebar:')
    print(f'  \ud83d\udcc1 Files \u2192 /content/periliminal_worlds.zip \u2192 Right-click \u2192 Download')

print()
print('='*50)
print('\ud83d\udccb INSTALL INSTRUCTIONS')
print('='*50)
print('1. Extract the zip into your Godot project folder')
print('   Copy the videos/ folder to: assets/references/lingbot/videos/')
print('2. Open Godot, press F5')
print('3. LingBotIntegration autoload picks them up automatically')

---
## Troubleshooting

| Problem | Fix |
|---------|-----|
| No GPU detected | Runtime → Change runtime type → T4 GPU, then restart |
| Colab disconnects | Free tier times out after ~90 min. Re-run Step 5 — skips completed videos |
| CUDA out of memory | `--offload_model True` is already set in the command |
| Model download fails | Re-run Step 2. If persists, download manually from Hugging Face |
| Godot shows "No loader found" for .ipynb | Harmless — it's a text file, not a game asset |
| Need to restart fresh | Runtime → Disconnect and delete runtime → Start over |